# Feature Engineering

## Categorisation of bets based on the timing of the match
Timing of bets can be placed into 3 major categories:

1. Before the ban/pick phase
2. After the ban/pick phase and before match has started
3. Anytime before the game has ended

For this EDA, we will focus on the first two categories and the last category can be left as a future extension of the project.

In [1]:
%load_ext autoreload
%autoreload 2
from sqlalchemy.engine import create_engine, URL
from datetime import datetime as dt
from utils import unix_to_datetime
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 100)
pd.set_option('display.max_rows', 50)



In [2]:
url_object = URL.create(
    "postgresql+psycopg2",
    username='liuhaochen',
    host='localhost',
    port='5432',
    database='test'
)

engine = create_engine(url_object)

In [3]:
df = pd.read_sql(
    "SELECT * FROM pro_matches",
    con=engine
)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58902 entries, 0 to 58901
Data columns (total 28 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   0_hero_id        58766 non-null  float64
 1   1_hero_id        58769 non-null  float64
 2   2_hero_id        58773 non-null  float64
 3   3_hero_id        58769 non-null  float64
 4   4_hero_id        58769 non-null  float64
 5   128_hero_id      58766 non-null  float64
 6   129_hero_id      58771 non-null  float64
 7   130_hero_id      58780 non-null  float64
 8   131_hero_id      58771 non-null  float64
 9   132_hero_id      58775 non-null  float64
 10  0_account_id     58766 non-null  float64
 11  1_account_id     58769 non-null  float64
 12  2_account_id     58773 non-null  float64
 13  3_account_id     58769 non-null  float64
 14  4_account_id     58769 non-null  float64
 15  128_account_id   58766 non-null  float64
 16  129_account_id   58771 non-null  float64
 17  130_account_

# Feature Selection (Manual)

In [4]:
draft_cols = df.filter(like="_hero_id").columns
player_cols = df.filter(like="_account_id").columns
team_cols = ['radiant_name','dire_name']
label_col = 'radiant_win'
time_col = 'start_time'
uuid_col = 'match_id'

In [5]:
def preprocess_df(df):
    final_cols = np.concatenate((draft_cols, player_cols, team_cols, [label_col],[time_col],[uuid_col]) )
    df = df[final_cols]
    df = df.dropna(axis=0)
    df['start_time'] = df['start_time'].apply(unix_to_datetime)
    df = df.drop_duplicates(subset='match_id').reset_index(drop=True)
    df.sort_values(by='start_time', ascending=False, inplace=True)
    
    return df

In [6]:
df = preprocess_df(df)
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 56485 entries, 56215 to 42597
Data columns (total 25 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   0_hero_id       56485 non-null  float64       
 1   1_hero_id       56485 non-null  float64       
 2   2_hero_id       56485 non-null  float64       
 3   3_hero_id       56485 non-null  float64       
 4   4_hero_id       56485 non-null  float64       
 5   128_hero_id     56485 non-null  float64       
 6   129_hero_id     56485 non-null  float64       
 7   130_hero_id     56485 non-null  float64       
 8   131_hero_id     56485 non-null  float64       
 9   132_hero_id     56485 non-null  float64       
 10  0_account_id    56485 non-null  float64       
 11  1_account_id    56485 non-null  float64       
 12  2_account_id    56485 non-null  float64       
 13  3_account_id    56485 non-null  float64       
 14  4_account_id    56485 non-null  float64       
 15

# Feature Engineering

## Create Team Level Features

In [7]:
def last_10_matches_win_rate(df, team_name, current_date):
    last_10_matches = df[((df['radiant_name'] == team_name) | (df['dire_name'] == team_name)) &
                         (df['start_time'] < current_date)].tail(10)
                         
    
    if len(last_10_matches) == 0:
        return 0.5 # Impute value as 0.5 for a team's debut match

    wins = ((last_10_matches['radiant_name'] == team_name) & last_10_matches['radiant_win']).sum()
    wins += ((last_10_matches['dire_name'] == team_name) & ~last_10_matches['radiant_win']).sum()

    win_rate = wins / len(last_10_matches)
    return win_rate

In [8]:

df['radiant_win_rate'] = df.apply(lambda row: last_10_matches_win_rate(df, row['radiant_name'], row['start_time']), axis=1)
df['dire_win_rate'] = df.apply(lambda row: last_10_matches_win_rate(df, row['dire_name'], row['start_time']), axis=1)


In [9]:
def radiant_dire_matchup(df, radiant_name, dire_name, current_date):
    last_10_matches = df[
        (((df['radiant_name'] == radiant_name) & (df['dire_name'] == dire_name)) |
         ((df['dire_name'] == radiant_name) & (df['radiant_name'] == dire_name))) &
        (df['start_time'] < current_date)].tail(10)
    
    if len(last_10_matches) == 0:
        return 0.5 # Impute value as 0.5 for a team's debut match
    
    wins = ((last_10_matches['radiant_name'] == radiant_name) & last_10_matches['radiant_win']).sum()
    wins += ((last_10_matches['dire_name'] == radiant_name) & ~last_10_matches['radiant_win']).sum()

    radiant_dire_matchup = wins / len(last_10_matches)
    
    return radiant_dire_matchup

In [10]:
df['radiant_dire_matchup'] = df.apply(lambda row: radiant_dire_matchup(df, row['radiant_name'], row['dire_name'], row['start_time']), axis=1)

In [11]:
team_level_features = df[['radiant_dire_matchup','radiant_win_rate','dire_win_rate',label_col,time_col,uuid_col]]

In [12]:
team_level_features

,radiant_dire_matchup,radiant_win_rate,dire_win_rate,radiant_win,start_time,match_id
56215,0.750000,1.0,0.7,True,2023-09-30 09:52:54,7.359096e+09
56216,0.900000,0.8,0.9,True,2023-09-30 09:37:19,7.359076e+09
56217,0.800000,0.9,0.8,False,2023-09-30 08:55:16,7.359003e+09
56218,1.000000,0.7,1.0,False,2023-09-30 08:51:13,7.358997e+09
56219,0.714286,0.9,0.8,True,2023-09-30 08:03:22,7.358893e+09
...,...,...,...,...,...,...
42593,0.500000,0.5,0.5,False,2021-05-17 17:02:06,5.999283e+09
42594,1.000000,1.0,0.0,False,2021-05-17 16:02:25,5.999250e+09
42595,0.500000,0.5,0.5,False,2021-05-17 15:02:01,5.999214e+09
42596,0.500000,0.5,0.5,True,2021-05-17 14:40:35,5.999202e+09


## Create hero level feature 

In [13]:
import yaml
CONSTANTS_FILE_PATH = 'constants.yml'
try:
    with open(CONSTANTS_FILE_PATH, 'r') as file:
        data = yaml.safe_load(file) or {}
        hero_dict = data.get('HEROES_CONSTANTS', {})
        if not hero_dict or not isinstance(hero_dict, dict):
            raise ValueError("Unable to load hero constants or they are not in a valid format.")
except FileNotFoundError:
    print(f"'{CONSTANTS_FILE_PATH}' does not exist!")
    
hero_dict

{1: 'Anti-Mage',
 2: 'Axe',
 3: 'Bane',
 4: 'Bloodseeker',
 5: 'Crystal Maiden',
 6: 'Drow Ranger',
 7: 'Earthshaker',
 8: 'Juggernaut',
 9: 'Mirana',
 10: 'Morphling',
 11: 'Shadow Fiend',
 12: 'Phantom Lancer',
 13: 'Puck',
 14: 'Pudge',
 15: 'Razor',
 16: 'Sand King',
 17: 'Storm Spirit',
 18: 'Sven',
 19: 'Tiny',
 20: 'Vengeful Spirit',
 21: 'Windranger',
 22: 'Zeus',
 23: 'Kunkka',
 25: 'Lina',
 26: 'Lion',
 27: 'Shadow Shaman',
 28: 'Slardar',
 29: 'Tidehunter',
 30: 'Witch Doctor',
 31: 'Lich',
 32: 'Riki',
 33: 'Enigma',
 34: 'Tinker',
 35: 'Sniper',
 36: 'Necrophos',
 37: 'Warlock',
 38: 'Beastmaster',
 39: 'Queen of Pain',
 40: 'Venomancer',
 41: 'Faceless Void',
 42: 'Wraith King',
 43: 'Death Prophet',
 44: 'Phantom Assassin',
 45: 'Pugna',
 46: 'Templar Assassin',
 47: 'Viper',
 48: 'Luna',
 49: 'Dragon Knight',
 50: 'Dazzle',
 51: 'Clockwerk',
 52: 'Leshrac',
 53: "Nature's Prophet",
 54: 'Lifestealer',
 55: 'Dark Seer',
 56: 'Clinkz',
 57: 'Omniknight',
 58: 'Enchantress

In [14]:
df[draft_cols] = df[draft_cols].applymap(hero_dict.get)
df[draft_cols]

,0_hero_id,1_hero_id,2_hero_id,3_hero_id,4_hero_id,128_hero_id,129_hero_id,130_hero_id,131_hero_id,132_hero_id
56215,Treant Protector,Pangolier,Muerta,Skywrath Mage,Beastmaster,Shadow Demon,Grimstroke,Tidehunter,Terrorblade,Ember Spirit
56216,Dark Willow,Lone Druid,Puck,Earth Spirit,Crystal Maiden,Oracle,Rubick,Lifestealer,Slardar,Dawnbreaker
56217,Lone Druid,Skywrath Mage,Vengeful Spirit,Ember Spirit,Monkey King,Bristleback,Earth Spirit,Clinkz,Huskar,Crystal Maiden
56218,Clockwerk,Phantom Assassin,Death Prophet,Kunkka,Skywrath Mage,Treant Protector,Timbersaw,Morphling,Grimstroke,Tidehunter
56219,Morphling,Primal Beast,Tidehunter,Skywrath Mage,Treant Protector,Dark Willow,Sven,Invoker,Doom,Earthshaker
...,...,...,...,...,...,...,...,...,...,...
42593,Enchantress,Timbersaw,Tiny,Wraith King,Oracle,Faceless Void,Ancient Apparition,Centaur Warrunner,Snapfire,Templar Assassin
42594,Void Spirit,Brewmaster,Terrorblade,Snapfire,Abaddon,Rubick,Centaur Warrunner,Ember Spirit,Medusa,Ancient Apparition
42595,Centaur Warrunner,Hoodwink,Razor,Grimstroke,Gyrocopter,Kunkka,Axe,Medusa,Snapfire,Witch Doctor
42596,Ancient Apparition,Enchantress,Magnus,Ember Spirit,Tiny,Oracle,Storm Spirit,Broodmother,Nyx Assassin,Drow Ranger


In [15]:
df_melted_heroes = pd.melt(df, id_vars=['start_time','radiant_win','match_id'],
                            value_vars=draft_cols, 
                            var_name='hero_position',
                            value_name='hero_name')

df_melted_heroes

,start_time,radiant_win,match_id,hero_position,hero_name
0,2023-09-30 09:52:54,True,7.359096e+09,0_hero_id,Treant Protector
1,2023-09-30 09:37:19,True,7.359076e+09,0_hero_id,Dark Willow
2,2023-09-30 08:55:16,False,7.359003e+09,0_hero_id,Lone Druid
3,2023-09-30 08:51:13,False,7.358997e+09,0_hero_id,Clockwerk
4,2023-09-30 08:03:22,True,7.358893e+09,0_hero_id,Morphling
...,...,...,...,...,...
564845,2021-05-17 17:02:06,False,5.999283e+09,132_hero_id,Templar Assassin
564846,2021-05-17 16:02:25,False,5.999250e+09,132_hero_id,Ancient Apparition
564847,2021-05-17 15:02:01,False,5.999214e+09,132_hero_id,Witch Doctor
564848,2021-05-17 14:40:35,True,5.999202e+09,132_hero_id,Drow Ranger


In [16]:
df_encoded = pd.concat([df_melted_heroes, pd.get_dummies(df_melted_heroes['hero_name'])], axis=1)

df_encoded = df_encoded.groupby(['match_id']).sum(numeric_only=True).reset_index()

cols_to_merge = ['start_time', 'radiant_win', 'match_id']
heroes_features = df_encoded.merge(df[cols_to_merge].drop_duplicates(), on='match_id', how='left')
heroes_features

,match_id,Abaddon,Alchemist,Ancient Apparition,Anti-Mage,Arc Warden,Axe,Bane,Batrider,Beastmaster,Bloodseeker,Bounty Hunter,Brewmaster,Bristleback,Broodmother,Centaur Warrunner,Chaos Knight,Chen,Clinkz,Clockwerk,Crystal Maiden,Dark Seer,Dark Willow,Dawnbreaker,Dazzle,Death Prophet,Disruptor,Doom,Dragon Knight,Drow Ranger,Earth Spirit,Earthshaker,Elder Titan,Ember Spirit,Enchantress,Enigma,Faceless Void,Grimstroke,Gyrocopter,Hoodwink,Huskar,Invoker,Io,Jakiro,Juggernaut,Keeper of the Light,Kunkka,Legion Commander,Leshrac,Lich,...,Phoenix,Primal Beast,Puck,Pudge,Pugna,Queen of Pain,Razor,Riki,Rubick,Sand King,Shadow Demon,Shadow Fiend,Shadow Shaman,Silencer,Skywrath Mage,Slardar,Slark,Snapfire,Sniper,Spectre,Spirit Breaker,Storm Spirit,Sven,Techies,Templar Assassin,Terrorblade,Tidehunter,Timbersaw,Tinker,Tiny,Treant Protector,Troll Warlord,Tusk,Underlord,Undying,Ursa,Vengeful Spirit,Venomancer,Viper,Visage,Void Spirit,Warlock,Weaver,Windranger,Winter Wyvern,Witch Doctor,Wraith King,Zeus,start_time,radiant_win
0,5.999176e+09,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,2021-05-17 14:06:51,True
1,5.999202e+09,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2021-05-17 14:40:35,True
2,5.999214e+09,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,2021-05-17 15:02:01,False
3,5.999250e+09,1,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,2021-05-17 16:02:25,False
4,5.999283e+09,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,2021-05-17 17:02:06,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56480,7.358893e+09,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2023-09-30 08:03:22,True
56481,7.358997e+09,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2023-09-30 08:51:13,False
56482,7.359003e+09,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,2023-09-30 08:55:16,False
56483,7.359076e+09,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2023-09-30 09:37:19,True


## Feature Crossing between players and heros



In [17]:
df_melted_players = pd.melt(df.copy(), id_vars=['start_time','radiant_win','match_id'],
                            value_vars=player_cols, 
                            var_name='player_position',
                            value_name='account_id')

df_melted_players

,start_time,radiant_win,match_id,player_position,account_id
0,2023-09-30 09:52:54,True,7.359096e+09,0_account_id,1.174215e+08
1,2023-09-30 09:37:19,True,7.359076e+09,0_account_id,9.170671e+08
2,2023-09-30 08:55:16,False,7.359003e+09,0_account_id,1.247302e+09
3,2023-09-30 08:51:13,False,7.358997e+09,0_account_id,1.050453e+08
4,2023-09-30 08:03:22,True,7.358893e+09,0_account_id,2.300986e+08
...,...,...,...,...,...
564845,2021-05-17 17:02:06,False,5.999283e+09,132_account_id,1.727400e+08
564846,2021-05-17 16:02:25,False,5.999250e+09,132_account_id,7.660036e+07
564847,2021-05-17 15:02:01,False,5.999214e+09,132_account_id,6.789384e+07
564848,2021-05-17 14:40:35,True,5.999202e+09,132_account_id,1.075799e+08


In [18]:
df_melted_heros = pd.melt(df.copy(), id_vars=['start_time','radiant_win','match_id'],
                            value_vars=draft_cols,
                            var_name='hero_position',
                            value_name='hero_id')

df_melted_heros

,start_time,radiant_win,match_id,hero_position,hero_id
0,2023-09-30 09:52:54,True,7.359096e+09,0_hero_id,Treant Protector
1,2023-09-30 09:37:19,True,7.359076e+09,0_hero_id,Dark Willow
2,2023-09-30 08:55:16,False,7.359003e+09,0_hero_id,Lone Druid
3,2023-09-30 08:51:13,False,7.358997e+09,0_hero_id,Clockwerk
4,2023-09-30 08:03:22,True,7.358893e+09,0_hero_id,Morphling
...,...,...,...,...,...
564845,2021-05-17 17:02:06,False,5.999283e+09,132_hero_id,Templar Assassin
564846,2021-05-17 16:02:25,False,5.999250e+09,132_hero_id,Ancient Apparition
564847,2021-05-17 15:02:01,False,5.999214e+09,132_hero_id,Witch Doctor
564848,2021-05-17 14:40:35,True,5.999202e+09,132_hero_id,Drow Ranger


In [19]:
def extract_num(position):
    return position.split('_')[0]

In [20]:
df_melted_players['player_num'] = df_melted_players['player_position'].apply(extract_num)
df_melted_heros['hero_num'] = df_melted_heros['hero_position'].apply(extract_num)

In [21]:
df_combined = pd.merge(df_melted_players, df_melted_heros, 
                       left_on=['start_time', 'radiant_win','match_id', 'player_num'], 
                       right_on=['start_time', 'radiant_win','match_id', 'hero_num'])

df_combined = df_combined.sort_values(by='start_time')
df_combined


,start_time,radiant_win,match_id,player_position,account_id,player_num,hero_position,hero_id,hero_num
282424,2021-05-17 14:06:51,True,5.999176e+09,4_account_id,413339999.0,4,4_hero_id,Dark Seer,4
508364,2021-05-17 14:06:51,True,5.999176e+09,131_account_id,194521913.0,131,131_hero_id,Timbersaw,131
451879,2021-05-17 14:06:51,True,5.999176e+09,130_account_id,101779337.0,130,130_hero_id,Earthshaker,130
395394,2021-05-17 14:06:51,True,5.999176e+09,129_account_id,173971950.0,129,129_hero_id,Lifestealer,129
338909,2021-05-17 14:06:51,True,5.999176e+09,128_account_id,116293223.0,128,128_hero_id,Batrider,128
...,...,...,...,...,...,...,...,...,...
225940,2023-09-30 09:52:54,True,7.359096e+09,4_account_id,228516683.0,4,4_hero_id,Beastmaster,4
395395,2023-09-30 09:52:54,True,7.359096e+09,130_account_id,138880576.0,130,130_hero_id,Tidehunter,130
282425,2023-09-30 09:52:54,True,7.359096e+09,128_account_id,105045291.0,128,128_hero_id,Shadow Demon,128
112970,2023-09-30 09:52:54,True,7.359096e+09,2_account_id,879017980.0,2,2_hero_id,Muerta,2


In [22]:
def calculate_win_rate(matches):
    num_matches = len(matches)
    if num_matches == 0:
        return 0.5 # impute win_rate to 0.5 
    
    wins = ((matches['player_num'].apply(int) < 5) & matches[label_col]).sum()
    wins += ((matches['player_num'].apply(int) > 5) & ~matches[label_col]).sum()
    
    return wins/num_matches

In [23]:
def last_10_matches_winrate(df, current_date, account_id, hero_id):
    df_filtered = df[(df['account_id']==account_id) & 
                     (df['hero_id']==hero_id) & 
                     (df[time_col] < current_date)].sort_values(
                         by=time_col, ascending=False)
    
    return calculate_win_rate(df_filtered[:10])

In [24]:
df_combined['win_rate'] = df_combined.apply(lambda row: last_10_matches_winrate(df_combined, row[time_col], row['account_id'], row['hero_id']), axis=1)
df_combined['win_rate']

282424    0.5
508364    0.5
451879    0.5
395394    0.5
338909    0.5
         ... 
225940    0.8
395395    0.8
282425    0.7
112970    0.5
0         0.8
Name: win_rate, Length: 564850, dtype: float64

In [25]:
df_combined

,start_time,radiant_win,match_id,player_position,account_id,player_num,hero_position,hero_id,hero_num,win_rate
282424,2021-05-17 14:06:51,True,5.999176e+09,4_account_id,413339999.0,4,4_hero_id,Dark Seer,4,0.5
508364,2021-05-17 14:06:51,True,5.999176e+09,131_account_id,194521913.0,131,131_hero_id,Timbersaw,131,0.5
451879,2021-05-17 14:06:51,True,5.999176e+09,130_account_id,101779337.0,130,130_hero_id,Earthshaker,130,0.5
395394,2021-05-17 14:06:51,True,5.999176e+09,129_account_id,173971950.0,129,129_hero_id,Lifestealer,129,0.5
338909,2021-05-17 14:06:51,True,5.999176e+09,128_account_id,116293223.0,128,128_hero_id,Batrider,128,0.5
...,...,...,...,...,...,...,...,...,...,...
225940,2023-09-30 09:52:54,True,7.359096e+09,4_account_id,228516683.0,4,4_hero_id,Beastmaster,4,0.8
395395,2023-09-30 09:52:54,True,7.359096e+09,130_account_id,138880576.0,130,130_hero_id,Tidehunter,130,0.8
282425,2023-09-30 09:52:54,True,7.359096e+09,128_account_id,105045291.0,128,128_hero_id,Shadow Demon,128,0.7
112970,2023-09-30 09:52:54,True,7.359096e+09,2_account_id,879017980.0,2,2_hero_id,Muerta,2,0.5


In [26]:
df_combined['player_hero_win_rate_col'] = df_combined['player_num'].astype(str) + '_account_ ' + df_combined['hero_num'].astype(str) + '_hero_win_rate'
player_hero_feature = df_combined.pivot(index='match_id', columns='player_hero_win_rate_col', values='win_rate').reset_index()
player_hero_feature

player_hero_win_rate_col,match_id,0_account_ 0_hero_win_rate,128_account_ 128_hero_win_rate,129_account_ 129_hero_win_rate,130_account_ 130_hero_win_rate,131_account_ 131_hero_win_rate,132_account_ 132_hero_win_rate,1_account_ 1_hero_win_rate,2_account_ 2_hero_win_rate,3_account_ 3_hero_win_rate,4_account_ 4_hero_win_rate
0,5.999176e+09,0.5,0.500000,0.500000,0.500,0.500,0.5,0.5,0.500,0.50,0.500
1,5.999202e+09,0.5,0.500000,0.500000,0.500,0.500,0.5,0.5,0.500,0.50,0.500
2,5.999214e+09,0.5,0.500000,0.500000,0.500,0.500,0.5,0.5,0.500,0.50,0.500
3,5.999250e+09,0.5,0.500000,0.000000,0.500,0.500,0.5,0.5,0.500,1.00,0.500
4,5.999283e+09,0.5,0.500000,0.500000,0.500,0.500,0.5,0.5,0.500,0.50,0.500
...,...,...,...,...,...,...,...,...,...,...,...
56480,7.358893e+09,0.7,0.900000,0.666667,1.000,0.625,1.0,0.8,0.600,1.00,0.875
56481,7.358997e+09,0.8,0.777778,0.800000,0.625,0.800,0.9,0.6,0.625,0.70,1.000
56482,7.359003e+09,0.7,0.750000,0.714286,0.900,0.500,0.7,0.7,0.600,0.50,0.500
56483,7.359076e+09,1.0,0.800000,0.700000,0.500,0.800,0.5,0.8,1.000,0.75,0.700


## Merging all features

In [27]:
df_final = pd.merge(pd.merge(team_level_features,heroes_features,how='inner', on=['match_id','radiant_win','start_time']),player_hero_feature, how='inner', on='match_id')
df_final

,radiant_dire_matchup,radiant_win_rate,dire_win_rate,radiant_win,start_time,match_id,Abaddon,Alchemist,Ancient Apparition,Anti-Mage,Arc Warden,Axe,Bane,Batrider,Beastmaster,Bloodseeker,Bounty Hunter,Brewmaster,Bristleback,Broodmother,Centaur Warrunner,Chaos Knight,Chen,Clinkz,Clockwerk,Crystal Maiden,Dark Seer,Dark Willow,Dawnbreaker,Dazzle,Death Prophet,Disruptor,Doom,Dragon Knight,Drow Ranger,Earth Spirit,Earthshaker,Elder Titan,Ember Spirit,Enchantress,Enigma,Faceless Void,Grimstroke,Gyrocopter,Hoodwink,Huskar,Invoker,Io,Jakiro,Juggernaut,...,Rubick,Sand King,Shadow Demon,Shadow Fiend,Shadow Shaman,Silencer,Skywrath Mage,Slardar,Slark,Snapfire,Sniper,Spectre,Spirit Breaker,Storm Spirit,Sven,Techies,Templar Assassin,Terrorblade,Tidehunter,Timbersaw,Tinker,Tiny,Treant Protector,Troll Warlord,Tusk,Underlord,Undying,Ursa,Vengeful Spirit,Venomancer,Viper,Visage,Void Spirit,Warlock,Weaver,Windranger,Winter Wyvern,Witch Doctor,Wraith King,Zeus,0_account_ 0_hero_win_rate,128_account_ 128_hero_win_rate,129_account_ 129_hero_win_rate,130_account_ 130_hero_win_rate,131_account_ 131_hero_win_rate,132_account_ 132_hero_win_rate,1_account_ 1_hero_win_rate,2_account_ 2_hero_win_rate,3_account_ 3_hero_win_rate,4_account_ 4_hero_win_rate
0,0.750000,1.0,0.7,True,2023-09-30 09:52:54,7.359096e+09,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.8,0.700000,0.600000,0.800,0.800,0.5,0.7,0.500,0.70,0.800
1,0.900000,0.8,0.9,True,2023-09-30 09:37:19,7.359076e+09,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1.0,0.800000,0.700000,0.500,0.800,0.5,0.8,1.000,0.75,0.700
2,0.800000,0.9,0.8,False,2023-09-30 08:55:16,7.359003e+09,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0.7,0.750000,0.714286,0.900,0.500,0.7,0.7,0.600,0.50,0.500
3,1.000000,0.7,1.0,False,2023-09-30 08:51:13,7.358997e+09,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.8,0.777778,0.800000,0.625,0.800,0.9,0.6,0.625,0.70,1.000
4,0.714286,0.9,0.8,True,2023-09-30 08:03:22,7.358893e+09,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.7,0.900000,0.666667,1.000,0.625,1.0,0.8,0.600,1.00,0.875
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56480,0.500000,0.5,0.5,False,2021-05-17 17:02:06,5.999283e+09,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0.5,0.500000,0.500000,0.500,0.500,0.5,0.5,0.500,0.50,0.500
56481,1.000000,1.0,0.0,False,2021-05-17 16:02:25,5.999250e+09,1,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0.5,0.500000,0.000000,0.500,0.500,0.5,0.5,0.500,1.00,0.500
56482,0.500000,0.5,0.5,False,2021-05-17 15:02:01,5.999214e+09,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0.5,0.500000,0.500000,0.500,0.500,0.5,0.5,0.500,0.50,0.500
56483,0.500000,0.5,0.5,True

In [28]:
df_final.to_csv("df_final.csv", index=False)